# Supreme Court Decision Prediction — Exploration Notebook
Visualize CPTs, run inference, and analyze model accuracy.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from network_structure import COLUMN_MAP, DISPOSITION_LABELS, TOPOLOGICAL_ORDER
from cpt_builder import CPTBuilder
from inference import RejectionSampler
from evaluate import top_k_accuracy, distribution_summary

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Data & Train

In [ ]:
DATA_PATH = '../data/SCDB_2023_01_caseCentered_Citation.csv'
df = pd.read_csv(DATA_PATH, encoding='latin-1', low_memory=False)
print(f'Loaded SCDB: {len(df)} cases, {len(df.columns)} columns')

# Train/test split
rng = np.random.default_rng(42)
idx = rng.permutation(len(df))
df_train = df.iloc[idx[30:]]
df_test  = df.iloc[idx[:30]]

builder = CPTBuilder(alpha=1.0).fit(df_train)
print(f'CPTs built for {len(builder.cpts)} nodes')

## 2. Outcome Distribution

In [ ]:
disp_col = COLUMN_MAP['final_disposition']
counts = df_train[disp_col].value_counts().sort_index()

labels = [f"{int(k)}: {DISPOSITION_LABELS.get(int(k), str(k))[:30]}" for k in counts.index]
probs  = counts.values / counts.values.sum()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(labels, probs, color='steelblue')
for bar, p in zip(bars, probs):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{p*100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Probability')
ax.set_title('Supreme Court Case Outcome Distribution (Training Set)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 3. Run Inference on a Single Case

In [ ]:
sampler = RejectionSampler(builder, random_state=42)

example_row = df_test.iloc[0]
from evaluate import build_evidence
evidence = build_evidence(example_row, exclude_var='final_disposition')
true_val = example_row[COLUMN_MAP['final_disposition']]
print(f'True disposition: {int(true_val)} — {DISPOSITION_LABELS.get(int(true_val), "?")}')
print(f'Evidence: {evidence}')

dist = sampler.query('final_disposition', evidence, n_samples=3000)

vals   = [k for k in sorted(dist, key=dist.get, reverse=True)]
probs_ = [dist[k] for k in vals]
lbls   = [f"{int(v)}: {DISPOSITION_LABELS.get(int(v), str(v))[:25]}" for v in vals]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['gold' if v == true_val else 'steelblue' for v in vals]
ax.barh(lbls, probs_, color=colors)
gold_patch = mpatches.Patch(color='gold', label='True outcome')
ax.legend(handles=[gold_patch])
ax.set_xlabel('Predicted Probability')
ax.set_title('Posterior Distribution — Single Case')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 4. Top-3 Accuracy on Test Set

In [ ]:
accuracy = top_k_accuracy(df_test, sampler, k=3, n_samples=1000, verbose=True)

fig, ax = plt.subplots(figsize=(5, 5))
sizes  = [accuracy, 1 - accuracy]
labels_ = [f'Correct\n{accuracy*100:.1f}%', f'Incorrect\n{(1-accuracy)*100:.1f}%']
colors_ = ['#4CAF50', '#F44336']
wedges, texts = ax.pie(sizes, labels=labels_, colors=colors_,
                        startangle=90, counterclock=False,
                        wedgeprops=dict(width=0.5))
ax.set_title('Supreme Court Prediction\nTop-3 Accuracy', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()